# Question → JSON plan translator: training on Colab

Fine-tunes a small model (LoRA) to turn a short question into a JSON plan, using `train.py` and generated training data.
The same run as the Kaggle notebook, for weeks when the Kaggle GPU quota has run out.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Training takes about 3.5 h; keep the tab open.
2. Nothing is saved to Google Drive unless you set `USE_DRIVE = True` in the first code cell. With it, a dropped
   session continues from the last checkpoint when you **Run all** again; without it, training starts over.
3. At the end one file downloads: `<run>-results.zip`, with `summary.txt` (the dev scores), the `*.predictions.jsonl`
   files (the app scores them) and the model.


In [ ]:
# Settings
MODEL = "Qwen/Qwen3-0.6B"
RUN_NAME = "qwen06-v12qat"
BATCH = 8
QAT = True                   # train against the phone's INT4 rounding (train.py --qat)
EPOCHS = 2                   # passes over the training rows
LR = 1e-4                    # learning rate (train.py default 1e-4)
MAX_STEPS = -1               # -1: all EPOCHS; a number stops earlier
USE_DRIVE = False            # True: checkpoints go to Google Drive, so a dropped session continues instead of starting over

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/plan-translator/{RUN_NAME}"
else:
    OUT = f"/content/runs/{RUN_NAME}"
SAVE = f"/content/{RUN_NAME}-results"   # only what's worth downloading goes here
os.makedirs(os.path.dirname(OUT), exist_ok=True)
os.makedirs(SAVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# The library versions train.py was tested with
!pip install -q transformers==5.22.0 trl==1.17.0 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0
!pip uninstall -q -y torchao

In [ ]:
# Scripts and data
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/fatihsokmen/plan-translator-colab /content/repo
!tar xzf /content/repo/data.tar.gz -C /content/repo && wc -l /content/repo/data/*/*.jsonl

In [ ]:
# Training. Prints the dev exact-plan rate at the end.
!cd /content/repo && python train.py --model {MODEL} --data data --out {OUT} --bs {BATCH} --save_steps 200 --epochs {EPOCHS} --max_steps {MAX_STEPS} --lr {LR} {"--qat" if QAT else ""} 2>&1 | tee -a {OUT}.log
assert os.path.exists(f"{OUT}/final"), "training did not finish: the error is in the output above"

In [ ]:
# The dev score (held-out wordings of this run's data), saved with the run
log = [l.strip() for l in open(f"{OUT}.log")]
new_dev = [l for l in log if l.startswith("dev: exact plan")][-1]
int4_dev = ([l for l in log if l.startswith("dev int4")] or [""])[-1]
open(f"{SAVE}/summary.txt", "w").write(f"{RUN_NAME}\n{new_dev}\n{int4_dev}\n")
print(open(f"{SAVE}/summary.txt").read())

In [ ]:
# Plans for the test and probe sets; the app scores these files (blind_v4 by counts only)
for name, prompts in [("real_like_v2", "data/test/real_like_v2.prompts.jsonl"), ("independent_v3", "data/test/independent_v3.prompts.jsonl"),
                      ("probe_combos_v1", "data/probe/combos_v1.prompts.jsonl"), ("independent_v3_typos", "data/probe/independent_v3_typos.prompts.jsonl"),
                      ("blind_v4", "data/test/blind_v4.prompts.jsonl"), ("shapes_v7", "data/probe/shapes_v7.prompts.jsonl"),
                      ("weak_v9", "data/probe/weak_v9.prompts.jsonl"), ("phone_v1", "data/probe/phone_v1.prompts.jsonl"),
                      ("gaps_v11", "data/probe/gaps_v11.prompts.jsonl"),
                      ("phone_v2", "data/probe/phone_v2.prompts.jsonl"),
                      ("phone_v2_before", "data/probe/phone_v2_before.prompts.jsonl")]:
    !cd /content/repo && python predict.py --model {OUT}/final --prompts {prompts} --out {SAVE}/{name}.predictions.jsonl
!ls -la {SAVE}

In [ ]:
# The model itself, as float16 (about 1.2 GB): finer than bfloat16, so the INT4 rounding the run trained for is kept
import torch, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
m = AutoModelForCausalLM.from_pretrained(f"{OUT}/final", dtype=torch.float16)
m.save_pretrained("/content/model_bf16")
AutoTokenizer.from_pretrained(f"{OUT}/final").save_pretrained("/content/model_bf16")
shutil.make_archive(f"{SAVE}/{RUN_NAME}", "zip", "/content/model_bf16")
!ls -la {SAVE}

In [ ]:
# One download with everything above (the model is about 1.2 GB; Colab deletes it when the session ends)
import shutil
from google.colab import files
shutil.make_archive(f"/content/{RUN_NAME}-results", "zip", SAVE)
files.download(f"/content/{RUN_NAME}-results.zip")
